# FORESIGHT — Rolling-Origin Model Testing

This notebook performs the **proper time-series evaluation** of:

1. Seasonal Naive
2. XGBoost
3. LightGBM

The evaluation uses **rolling-origin backtesting** with an 8-week forecast horizon.

It reports:
- WAPE
- Forecast Bias
- Fold-by-fold results
- Average WAPE
- Model ranking
- Actual vs forecast plots

> This is the model-testing stage. The final production model should only be selected after reviewing these results.


In [ ]:
from pathlib import Path
import warnings
import time
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================

BASE_DIR = Path.cwd()

# If this notebook is inside src/notebooks, use:
# BASE_DIR = Path.cwd().parent.parent

DATA_DIR = BASE_DIR / "data" / "processed"
MODEL_DIR = BASE_DIR / "models"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILE = DATA_DIR / "analysis_ready_daily.csv"

# Forecast horizon required by the project:
# 8 weeks is used here.
HORIZON = 8

# Rolling-origin training endpoints.
# These dates must be Mondays because week_start is Monday.
FOLD_TRAIN_ENDS = [
    "2024-06-24",
    "2024-12-23",
    "2025-06-23",
    "2025-10-27",
]

# Use fewer trees for a faster evaluation.
# Increase to 500 for the final experiment if runtime is acceptable.
N_ESTIMATORS = 300

RANDOM_STATE = 42

print("Input:", INPUT_FILE)
print("Exists:", INPUT_FILE.exists())
print("Forecast horizon:", HORIZON, "weeks")
print("Folds:", FOLD_TRAIN_ENDS)


## 1. Load and prepare weekly SKU demand

In [ ]:
df = pd.read_csv(
    INPUT_FILE,
    parse_dates=["date"]
)

print("Daily shape:", df.shape)
print("SKUs:", df["sku_id"].nunique())
print("Date range:", df["date"].min(), "to", df["date"].max())

# Monday-starting weekly periods
df["week_start"] = (
    df["date"]
    .dt.to_period("W-SUN")
    .dt.start_time
)

weekly = (
    df.groupby(
        ["sku_id", "week_start"],
        as_index=False
    )
    .agg(
        units_sold=("units_sold", "sum"),
        revenue=("revenue", "sum"),
        avg_unit_price=("avg_unit_price", "mean"),
        avg_discount_pct=("avg_discount_pct", "mean"),
        transaction_count=("transaction_count", "sum")
    )
    .sort_values(["sku_id", "week_start"])
    .reset_index(drop=True)
)

print("Observed weekly rows:", len(weekly))
display(weekly.head())


## 2. Complete the SKU-week grid

A forecasting model needs a consistent weekly time axis.

For each SKU:
- periods before its first observed sale are not treated as demand;
- missing weeks after its first observed sale are treated as zero demand in this initial experiment.

This assumption should be documented and later checked against inventory/availability information.


In [ ]:
all_weeks = pd.date_range(
    weekly["week_start"].min(),
    weekly["week_start"].max(),
    freq="W-MON"
)

all_skus = weekly["sku_id"].unique()

complete_index = pd.MultiIndex.from_product(
    [all_skus, all_weeks],
    names=["sku_id", "week_start"]
)

weekly = (
    weekly
    .set_index(["sku_id", "week_start"])
    .reindex(complete_index)
    .reset_index()
)

first_sale = (
    df.groupby("sku_id")["date"]
    .min()
    .rename("first_sale_date")
)

weekly = weekly.merge(
    first_sale,
    on="sku_id",
    how="left"
)

before_first_sale = (
    weekly["week_start"] < weekly["first_sale_date"]
)

after_first_sale = ~before_first_sale

weekly.loc[before_first_sale, "units_sold"] = np.nan

weekly.loc[after_first_sale, "units_sold"] = (
    weekly.loc[after_first_sale, "units_sold"]
    .fillna(0)
)

weekly["revenue"] = weekly["revenue"].fillna(0)
weekly["avg_discount_pct"] = weekly["avg_discount_pct"].fillna(0)
weekly["transaction_count"] = weekly["transaction_count"].fillna(0)

weekly["avg_unit_price"] = (
    weekly
    .groupby("sku_id")["avg_unit_price"]
    .transform(lambda x: x.ffill().bfill())
)

weekly = weekly.drop(columns=["first_sale_date"])

weekly = weekly.sort_values(
    ["sku_id", "week_start"]
).reset_index(drop=True)

print("Complete weekly rows:", len(weekly))


## 3. Feature engineering

In [ ]:
weekly["year"] = weekly["week_start"].dt.year
weekly["month"] = weekly["week_start"].dt.month
weekly["quarter"] = weekly["week_start"].dt.quarter
weekly["week_of_year"] = (
    weekly["week_start"]
    .dt.isocalendar()
    .week
    .astype(int)
)

weekly["trend"] = (
    (weekly["week_start"] - weekly["week_start"].min())
    .dt.days // 7
)

weekly = weekly.sort_values(
    ["sku_id", "week_start"]
)

group = weekly.groupby("sku_id")

# Lag features
for lag in [1, 2, 4, 8, 13, 26]:
    weekly[f"lag_{lag}"] = (
        group["units_sold"].shift(lag)
    )

# Rolling features.
# shift(1) prevents the current target from leaking into its features.
group_demand = weekly.groupby("sku_id")["units_sold"]

for window in [4, 8, 13]:
    weekly[f"rolling_mean_{window}"] = (
        group_demand
        .shift(1)
        .rolling(window)
        .mean()
        .reset_index(level=0, drop=True)
    )

for window in [4, 8]:
    weekly[f"rolling_std_{window}"] = (
        group_demand
        .shift(1)
        .rolling(window)
        .std()
        .reset_index(level=0, drop=True)
    )

feature_columns = [
    "lag_1",
    "lag_2",
    "lag_4",
    "lag_8",
    "lag_13",
    "lag_26",
    "rolling_mean_4",
    "rolling_mean_8",
    "rolling_mean_13",
    "rolling_std_4",
    "rolling_std_8",
    "avg_unit_price",
    "avg_discount_pct",
    "month",
    "quarter",
    "week_of_year",
    "trend"
]

weekly_model = weekly.dropna(
    subset=feature_columns
).copy()

# Global SKU encoding
sku_categories = pd.Categorical(
    weekly_model["sku_id"]
)

weekly_model["sku_code"] = sku_categories.codes

features = [
    "sku_code",
    *feature_columns
]

print("Model rows:", len(weekly_model))
print("Features:", len(features))


## 4. Metrics

In [ ]:
def wape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    denominator = np.sum(np.abs(y_true))

    if denominator == 0:
        return np.nan

    return np.sum(
        np.abs(y_true - y_pred)
    ) / denominator


def forecast_bias(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    return np.mean(y_pred - y_true)


def safe_predict(model, X):
    pred = model.predict(X)

    # Demand cannot be negative.
    return np.maximum(pred, 0)


## 5. Rolling-origin backtest

For every fold:

1. Train only on data available before the forecast origin.
2. Forecast the next 8 weeks.
3. Calculate WAPE and bias.
4. Move the origin forward.
5. Repeat.

No future observations are used to train an earlier fold.


In [ ]:
fold_results = []
all_predictions = []

for fold_number, train_end_string in enumerate(
    FOLD_TRAIN_ENDS,
    start=1
):

    train_end = pd.Timestamp(train_end_string)

    # Next 8 weekly periods after training cutoff
    future_weeks = pd.date_range(
        train_end + pd.Timedelta(weeks=1),
        periods=HORIZON,
        freq="W-MON"
    )

    test_start = future_weeks.min()
    test_end = future_weeks.max()

    train = weekly_model[
        weekly_model["week_start"] <= train_end
    ].copy()

    test = weekly_model[
        weekly_model["week_start"].isin(future_weeks)
    ].copy()

    # Some SKUs may not have observations in every future week.
    # Evaluation is performed on available SKU-week rows.
    if len(test) == 0:
        print(f"Fold {fold_number}: no test data. Skipping.")
        continue

    X_train = train[features]
    y_train = train["units_sold"]

    X_test = test[features]
    y_test = test["units_sold"]

    print("\n" + "=" * 70)
    print(f"FOLD {fold_number}")
    print("=" * 70)
    print("Training end:", train_end.date())
    print("Test:", test_start.date(), "→", test_end.date())
    print("Train rows:", len(train))
    print("Test rows:", len(test))

    # --------------------------------------------------------
    # Seasonal Naive
    # --------------------------------------------------------

    baseline_pred = np.maximum(
        test["lag_1"].values,
        0
    )

    baseline_wape = wape(
        y_test,
        baseline_pred
    )

    baseline_bias = forecast_bias(
        y_test,
        baseline_pred
    )

    # --------------------------------------------------------
    # XGBoost
    # --------------------------------------------------------

    start_time = time.time()

    xgb_model = XGBRegressor(
        n_estimators=N_ESTIMATORS,
        learning_rate=0.05,
        max_depth=8,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        eval_metric="mae",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    xgb_model.fit(
        X_train,
        y_train,
        verbose=False
    )

    xgb_pred = safe_predict(
        xgb_model,
        X_test
    )

    xgb_time = time.time() - start_time

    xgb_wape = wape(
        y_test,
        xgb_pred
    )

    xgb_bias = forecast_bias(
        y_test,
        xgb_pred
    )

    # --------------------------------------------------------
    # LightGBM
    # --------------------------------------------------------

    start_time = time.time()

    lgb_model = LGBMRegressor(
        n_estimators=N_ESTIMATORS,
        learning_rate=0.05,
        num_leaves=63,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1
    )

    lgb_model.fit(
        X_train,
        y_train
    )

    lgb_pred = safe_predict(
        lgb_model,
        X_test
    )

    lgb_time = time.time() - start_time

    lgb_wape = wape(
        y_test,
        lgb_pred
    )

    lgb_bias = forecast_bias(
        y_test,
        lgb_pred
    )

    # --------------------------------------------------------
    # Save fold results
    # --------------------------------------------------------

    fold_results.extend([
        {
            "fold": fold_number,
            "model": "Seasonal Naive",
            "train_end": train_end,
            "test_start": test_start,
            "test_end": test_end,
            "wape": baseline_wape,
            "bias": baseline_bias,
            "training_seconds": 0
        },
        {
            "fold": fold_number,
            "model": "XGBoost",
            "train_end": train_end,
            "test_start": test_start,
            "test_end": test_end,
            "wape": xgb_wape,
            "bias": xgb_bias,
            "training_seconds": xgb_time
        },
        {
            "fold": fold_number,
            "model": "LightGBM",
            "train_end": train_end,
            "test_start": test_start,
            "test_end": test_end,
            "wape": lgb_wape,
            "bias": lgb_bias,
            "training_seconds": lgb_time
        }
    ])

    # Store predictions for later plots.
    fold_pred = test[
        ["sku_id", "week_start", "units_sold"]
    ].copy()

    fold_pred["fold"] = fold_number
    fold_pred["seasonal_naive"] = baseline_pred
    fold_pred["xgboost"] = xgb_pred
    fold_pred["lightgbm"] = lgb_pred

    all_predictions.append(fold_pred)

    print(f"Seasonal Naive WAPE: {baseline_wape:.4f}")
    print(f"XGBoost WAPE:       {xgb_wape:.4f}")
    print(f"LightGBM WAPE:      {lgb_wape:.4f}")


## 6. Fold-by-fold results

In [ ]:
results = pd.DataFrame(fold_results)

display(
    results.sort_values(
        ["fold", "wape"]
    )
)


## 7. Overall model comparison

In [ ]:
summary = (
    results
    .groupby("model", as_index=False)
    .agg(
        mean_wape=("wape", "mean"),
        median_wape=("wape", "median"),
        std_wape=("wape", "std"),
        mean_bias=("bias", "mean"),
        mean_training_seconds=("training_seconds", "mean")
    )
    .sort_values("mean_wape")
    .reset_index(drop=True)
)

display(summary)

best_model = summary.iloc[0]["model"]

print("\nBest model by mean WAPE:", best_model)

baseline_mean = summary.loc[
    summary["model"] == "Seasonal Naive",
    "mean_wape"
].iloc[0]

best_ml_rows = summary[
    summary["model"].isin(["XGBoost", "LightGBM"])
]

best_ml_mean = best_ml_rows["mean_wape"].min()

if best_ml_mean < baseline_mean:
    improvement = (
        (baseline_mean - best_ml_mean)
        / baseline_mean
        * 100
    )

    print(
        f"Best ML model improves mean WAPE by "
        f"{improvement:.2f}% versus Seasonal Naive."
    )
else:
    print(
        "Neither ML model beats Seasonal Naive "
        "on mean rolling-origin WAPE."
    )


## 8. WAPE by fold

In [ ]:
pivot_wape = results.pivot(
    index="fold",
    columns="model",
    values="wape"
)

display(pivot_wape)

pivot_wape.plot(
    kind="bar",
    figsize=(12, 6)
)

plt.title("Rolling-Origin WAPE by Fold")
plt.xlabel("Fold")
plt.ylabel("WAPE")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 9. Bias by fold

In [ ]:
pivot_bias = results.pivot(
    index="fold",
    columns="model",
    values="bias"
)

display(pivot_bias)

pivot_bias.plot(
    kind="bar",
    figsize=(12, 6)
)

plt.title("Rolling-Origin Forecast Bias by Fold")
plt.xlabel("Fold")
plt.ylabel("Mean Prediction Error")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 10. Inspect predictions for one SKU

This plot uses the last available backtest fold and compares the three approaches.


In [ ]:
predictions = pd.concat(
    all_predictions,
    ignore_index=True
)

last_fold = predictions["fold"].max()

last_fold_predictions = predictions[
    predictions["fold"] == last_fold
].copy()

example_sku = (
    last_fold_predictions["sku_id"]
    .iloc[0]
)

plot_df = last_fold_predictions[
    last_fold_predictions["sku_id"] == example_sku
].sort_values("week_start")

plt.figure(figsize=(14, 6))

plt.plot(
    plot_df["week_start"],
    plot_df["units_sold"],
    marker="o",
    label="Actual"
)

plt.plot(
    plot_df["week_start"],
    plot_df["seasonal_naive"],
    marker="o",
    label="Seasonal Naive"
)

plt.plot(
    plot_df["week_start"],
    plot_df["xgboost"],
    marker="o",
    label="XGBoost"
)

plt.plot(
    plot_df["week_start"],
    plot_df["lightgbm"],
    marker="o",
    label="LightGBM"
)

plt.title(
    f"Rolling Backtest — {example_sku} — Fold {last_fold}"
)

plt.xlabel("Week")
plt.ylabel("Units Sold")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 11. Save backtest results

These files can later be used in the project report/dashboard.


In [ ]:
results_file = (
    MODEL_DIR /
    "rolling_origin_fold_results.csv"
)

summary_file = (
    MODEL_DIR /
    "rolling_origin_model_summary.csv"
)

predictions_file = (
    MODEL_DIR /
    "rolling_origin_predictions.csv"
)

results.to_csv(
    results_file,
    index=False
)

summary.to_csv(
    summary_file,
    index=False
)

predictions.to_csv(
    predictions_file,
    index=False
)

print("Saved:")
print(results_file)
print(summary_file)
print(predictions_file)


# Interpretation

Use the **mean WAPE across rolling-origin folds** as the primary model-selection metric.

A good final decision is:

- If XGBoost or LightGBM consistently beats Seasonal Naive → select the better ML model.
- If an ML model wins only on one fold but loses badly on others → investigate stability.
- If neither ML model beats Seasonal Naive → retain/report Seasonal Naive rather than forcing a complex model.
- Use bias as a secondary diagnostic to identify systematic over- or under-forecasting.

The next project stage after this is the **final 6–8 week forecasting pipeline and inventory risk engine**.
